# Experiment 03: Physical GPU VRAM Profiling & Generative Benchmarking with Inactive Tucker (Notebook 02 Method - Kaggle Version)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Tasks**:
1. **GLUE MNLI Classification Benchmark**: Evaluated across **1,000 samples** (`validation_matched`).
2. **Open-Ended Generative Quality Test**: Prompt: *"What is the best recipe to make a chocolate cake?"* evaluated qualitatively with live token generation.

### Objectives & Physical Memory Elimination:
In previous iterations, inactive weights were factorized using DBSCAN clustering into 10 chunks with `[6, 160, 500]` ranks, which resulted in an 84% reconstruction error and degraded downstream generation. In this benchmark, we apply the **Notebook 02 Method directly for inactive weights**:

1. **Notebook 02 Inactive Factorization**:
   - Inactive coordinates ($4,500$ coordinates/matrix, $351,000$ model-wide) are isolated by activation variance, quarantining the 2,412 active coordinates in pristine FP32.
   - Processed via the exact Notebook 02 pipeline:
     - **SVD 95% Spectral Energy Denoising**
     - **60th Percentile Magnitude Sparsification (Zero-Masking)**
     - **4D Balanced Tucker Decomposition**: Reshapes $[4500, 1152]$ into $[45, 100, 24, 48]$ and decomposes with ranks `[30, 45, 16, 32]`.
2. **Physical GPU VRAM Reduction**:
   - Standard dense matrices ($6912 \times 1152$ and $1152 \times 6912$) are structurally replaced on GPU with `TuckerFactorizedInactiveRowLinear` and `TuckerFactorizedInactiveColLinear`.
   - Stores ONLY:
     - Uncompressed Active Weights ($2,412$ coordinates)
     - Compact 4D Tucker core $\mathcal{S} \in \mathbb{R}^{30 \times 45 \times 16 \times 32}$ ($691,200$ params) and 4 factor matrices ($7,770$ params).
   - Permanently eliminates **349,832,340 parameters** (**$\approx 1,334.50\text{ MB} = 1.33\text{ GB}$ physical GPU memory reduction** in FP32).
3. **Empirical Memory Metrics Profiled**:
   - **Static Parameter VRAM (MB)**: Exact GPU memory allocated to model parameters before and after factorization.
   - **Peak Runtime VRAM (MB)**: Maximum peak memory reached during live 1,000-sample inference (`torch.cuda.max_memory_allocated()`).
   - **Downstream Accuracy & Generative Coherence**: Evaluated on 1,000 GLUE MNLI samples and full 256-token chocolate cake recipe generation.

In [ ]:
# Optional: Install required dependencies if not already present in your Kaggle environment
!pip install -q tensorly datasets scikit-learn


In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Standard Imports
# =====================================================================
import os
import sys
import time
import gc
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

print("Environment configured.")
print("TensorLy Backend:", tl.get_backend())
print("PyTorch Version: ", torch.__version__)
print("CUDA Available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:     ", torch.cuda.get_device_name(0))


In [ ]:
# =====================================================================
# STEP 2: VRAM Profiling Utilities
# =====================================================================
def get_model_param_vram_mb(model):
    """Calculates physical VRAM consumed by all parameters currently on GPU (in MB)."""
    total_bytes = 0
    for p in model.parameters():
        if p.is_cuda:
            total_bytes += p.numel() * p.element_size()
    for b in model.buffers():
        if b.is_cuda:
            total_bytes += b.numel() * b.element_size()
    return total_bytes / (1024 ** 2)

def get_cuda_memory_snapshot():
    """Captures allocated, reserved, and peak GPU memory metrics (in MB)."""
    if not torch.cuda.is_available():
        return {"allocated_mb": 0.0, "reserved_mb": 0.0, "max_allocated_mb": 0.0}
    return {
        "allocated_mb": torch.cuda.memory_allocated() / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved() / (1024 ** 2),
        "max_allocated_mb": torch.cuda.max_memory_allocated() / (1024 ** 2),
    }

def clear_cuda_cache():
    """Flushes cached GPU memory and resets peak memory counters."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

print("VRAM profiling helper functions initialized.")


In [ ]:
# =====================================================================
# STEP 3: Load Gemma 3 1B IT & 1,000 GLUE MNLI Validation Samples
# =====================================================================
import huggingface_hub

MODEL_ID = "google/gemma-3-1b-it"
NUM_LAYERS = 26
HIDDEN_DIM = 1152
INTERMEDIATE_DIM = 6912
NUM_EVAL_SAMPLES = 1000

# Hugging Face Authentication for Gated Gemma-3 Model
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Logged into Hugging Face via User Secrets.")
else:
    print("Warning: No HF_TOKEN found. Ensure model is cached or provide token.")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} directly with AutoModelForCausalLM...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token=hf_token,
)
model.eval()

# Load 1,000 GLUE MNLI validation samples
ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = ds.select(range(NUM_EVAL_SAMPLES))
labels_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [
    tokenizer.encode(" " + name, add_special_tokens=False)[0]
    for name in labels_names
]

print(f"Loaded {MODEL_ID} across {NUM_LAYERS} decoder layers.")
print(f"GLUE MNLI Benchmark: {len(eval_data)} samples loaded.")
print(f"Candidate token IDs for labels ({labels_names}): {label_token_ids}")


In [ ]:
# =====================================================================
# STEP 4: Pristine Model 1,000-Sample Inference & Recipe Generation
# =====================================================================
clear_cuda_cache()

# Cache pristine weights on CPU
W_orig_all = {}
for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp
    W_orig_all[l] = {
        "gate_proj": lmod.gate_proj.weight.data.clone().cpu(),
        "up_proj": lmod.up_proj.weight.data.clone().cpu(),
        "down_proj": lmod.down_proj.weight.data.clone().cpu(),
    }

unprocessed_param_vram = get_model_param_vram_mb(model)
print(f"Initial Static Parameter VRAM (Pristine): {unprocessed_param_vram:.2f} MB")

# Profile activations for coordinate partition
layer_acts = {l: {"gate_proj": []} for l in range(NUM_LAYERS)}
hooks = []

for l in range(NUM_LAYERS):
    mlp = model.model.layers[l].mlp
    def make_hook(layer_idx):
        def hook(m, inp, out):
            layer_acts[layer_idx]["gate_proj"].append(out.detach().cpu().squeeze(0).mean(dim=0))
        return hook
    hooks.append(mlp.act_fn.register_forward_hook(make_hook(l)))

pristine_preds = []
ground_truths = []
model.eval()
print(f"Running Pristine Model GLUE MNLI evaluation across {NUM_EVAL_SAMPLES} samples...")

with torch.no_grad():
    for sample in tqdm(eval_data, desc="Pristine MNLI Inference (1000 samples)"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        pristine_preds.append(pred_label)
        ground_truths.append(sample["label"])

for h in hooks:
    h.remove()

pristine_snapshot = get_cuda_memory_snapshot()
pristine_accuracy = accuracy_score(ground_truths, pristine_preds)
pristine_peak_vram = pristine_snapshot["max_allocated_mb"]

print(f"\nPristine Uncompressed Baseline Results:")
print(f"  Downstream Accuracy (1,000 samples): {pristine_accuracy * 100:.2f}%")
print(f"  Static Parameter VRAM:               {unprocessed_param_vram:.2f} MB")
print(f"  Peak Runtime Inference VRAM:         {pristine_peak_vram:.2f} MB")

# Activation variance for coordinate partition
acts_var_all = {
    l: np.var(torch.stack(layer_acts[l]["gate_proj"][:50], dim=0).numpy(), axis=0)
    for l in range(NUM_LAYERS)
}

# --- Qualitative Generation Test: Chocolate Cake Recipe ---
print(f"\n{'='*95}")
print("Generating Recipe with Pristine Uncompressed Model: 'What is the best recipe to make a chocolate cake?'")
print(f"{'='*95}")

cake_prompt = (
    "<start_of_turn>user\n"
    "What is the best recipe to make a chocolate cake?<end_of_turn>\n"
    "<start_of_turn>model\n"
)
cake_inputs = tokenizer(cake_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    pristine_cake_tokens = model.generate(
        **cake_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
pristine_cake_recipe = tokenizer.decode(
    pristine_cake_tokens[0][cake_inputs.input_ids.shape[1]:], skip_special_tokens=True
)
print(pristine_cake_recipe)
print(f"{'='*95}")


In [ ]:
# =====================================================================
# STEP 5: Inactive Coordinate Partition (Notebook 02 Method)
# =====================================================================
NUM_INACTIVE = 4500
NUM_ACTIVE = 2412 # 6,912 - 4,500 = 2,412 active coordinates preserved in FP32

layer_coords = {}
for l in range(NUM_LAYERS):
    v = acts_var_all[l]
    sorted_idx = np.argsort(v)
    inactive_idx = sorted_idx[:NUM_INACTIVE]
    active_idx = sorted_idx[NUM_INACTIVE:]
    layer_coords[l] = {
        "inactive_indices": inactive_idx,
        "active_indices": active_idx,
    }

print(f"Partitioned coordinates: {NUM_INACTIVE} inactive / {NUM_ACTIVE} active per submodule across all {NUM_LAYERS} layers.")


In [ ]:
# =====================================================================
# STEP 6: Define Custom Physically Factorized Inactive Linear Modules (Notebook 02 Method)
# =====================================================================
class TuckerFactorizedInactiveRowLinear(nn.Module):
    """
    Physically factorized linear module for row-sliced projections (gate_proj, up_proj)
    using the Notebook 02 method directly for inactive weights.
    Eliminates 4,485,030 dense parameters per projection from GPU memory.
    Stores only:
      - active_weights: uncompressed FP32 (2,412 coords)
      - core: [30, 45, 16, 32] Tucker core (691,200 params)
      - factors: [45, 30], [100, 45], [24, 16], [48, 32] factor matrices (7,770 params)
    """
    def __init__(self, W_orig, active_indices, inactive_indices, core, factors):
        super().__init__()
        self.out_features, self.in_features = W_orig.shape
        self.register_buffer("active_indices", torch.tensor(active_indices, dtype=torch.long))
        self.register_buffer("inactive_indices", torch.tensor(inactive_indices, dtype=torch.long))
        
        self.active_weights = nn.Parameter(W_orig[active_indices, :].clone(), requires_grad=False)
        self.core = nn.Parameter(core.clone(), requires_grad=False)
        self.factors = nn.ParameterList([
            nn.Parameter(f.clone(), requires_grad=False) for f in factors
        ])

    def forward(self, x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, self.in_features)
        out = torch.empty((x_2d.shape[0], self.out_features), device=x.device, dtype=x.dtype)
        out[:, self.active_indices] = (x_2d @ self.active_weights.T).to(dtype=out.dtype)
        W_inact = tucker_to_tensor((self.core, list(self.factors))).reshape(-1, self.in_features)
        out[:, self.inactive_indices] = (x_2d @ W_inact.T).to(dtype=out.dtype)
        return out.reshape(*orig_shape[:-1], self.out_features)


class TuckerFactorizedInactiveColLinear(nn.Module):
    """
    Physically factorized linear module for column-sliced projections (down_proj)
    using the Notebook 02 method directly for inactive weights.
    Eliminates 4,485,030 dense parameters per projection from GPU memory.
    Stores only:
      - active_weights: uncompressed FP32 (2,412 coords)
      - core: [30, 45, 16, 32] Tucker core (691,200 params)
      - factors: [45, 30], [100, 45], [24, 16], [48, 32] factor matrices (7,770 params)
    """
    def __init__(self, W_orig, active_indices, inactive_indices, core, factors):
        super().__init__()
        self.out_features, self.in_features = W_orig.shape
        self.register_buffer("active_indices", torch.tensor(active_indices, dtype=torch.long))
        self.register_buffer("inactive_indices", torch.tensor(inactive_indices, dtype=torch.long))
        
        self.active_weights = nn.Parameter(W_orig[:, active_indices].clone(), requires_grad=False)
        self.core = nn.Parameter(core.clone(), requires_grad=False)
        self.factors = nn.ParameterList([
            nn.Parameter(f.clone(), requires_grad=False) for f in factors
        ])

    def forward(self, x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, self.in_features)
        W_inact = tucker_to_tensor((self.core, list(self.factors))).reshape(-1, self.out_features)
        out = (
            x_2d[:, self.active_indices] @ self.active_weights.T +
            x_2d[:, self.inactive_indices] @ W_inact
        )
        return out.reshape(*orig_shape[:-1], self.out_features)

print("Defined TuckerFactorizedInactiveRowLinear and TuckerFactorizedInactiveColLinear.")


In [ ]:
# =====================================================================
# STEP 7: Structural Replacement Across All 26 Layers (Notebook 02 Method)
# =====================================================================
RANKS_02 = [30, 45, 16, 32]

def factorize_inactive_02(W_inact, ranks=RANKS_02):
    # 1. SVD 95% Energy Denoising
    U, S, Vh = torch.linalg.svd(W_inact, full_matrices=False)
    cum_e = torch.cumsum(S**2, dim=0) / torch.sum(S**2)
    r95 = (cum_e >= 0.95).nonzero()[0].item() + 1
    W_denoised = U[:, :r95] @ torch.diag(S[:r95]) @ Vh[:r95, :]
    
    # 2. 60th Percentile Sparsification
    eps = torch.quantile(torch.abs(W_denoised), 0.60)
    W_sparse = W_denoised.clone()
    W_sparse[torch.abs(W_sparse) < eps] = 0.0
    
    # 3. 4D Tucker Decomposition: [4500, 1152] -> [45, 100, 24, 48]
    W_tensor = W_sparse.reshape(45, 100, 24, 48)
    core, factors = tucker(W_tensor, rank=ranks, init='svd')
    return core.detach(), [f.detach() for f in factors]

print(f"Applying physical structural replacement across all {NUM_LAYERS} layers (78 projections)...")

for l in tqdm(range(NUM_LAYERS), desc="Replacing Dense Layers with Notebook 02 Factorized Modules"):
    layer_mod = model.model.layers[l].mlp
    coords = layer_coords[l]
    act_idx = coords["active_indices"]
    inact_idx = coords["inactive_indices"]
    sub_dev = layer_mod.gate_proj.weight.device
    
    # 1. gate_proj
    W_gate = W_orig_all[l]["gate_proj"].to(sub_dev)
    cg, fg = factorize_inactive_02(W_gate[inact_idx, :].float())
    fact_gate = TuckerFactorizedInactiveRowLinear(W_gate, act_idx, inact_idx, cg, fg).to(sub_dev)
    del layer_mod.gate_proj
    layer_mod.gate_proj = fact_gate
    
    # 2. up_proj
    W_up = W_orig_all[l]["up_proj"].to(sub_dev)
    cu, fu = factorize_inactive_02(W_up[inact_idx, :].float())
    fact_up = TuckerFactorizedInactiveRowLinear(W_up, act_idx, inact_idx, cu, fu).to(sub_dev)
    del layer_mod.up_proj
    layer_mod.up_proj = fact_up
    
    # 3. down_proj
    W_down = W_orig_all[l]["down_proj"].to(sub_dev)
    cd, fd = factorize_inactive_02(W_down[:, inact_idx].T.float())
    fact_down = TuckerFactorizedInactiveColLinear(W_down, act_idx, inact_idx, cd, fd).to(sub_dev)
    del layer_mod.down_proj
    layer_mod.down_proj = fact_down

clear_cuda_cache()

factorized_param_vram = get_model_param_vram_mb(model)
static_vram_saved = unprocessed_param_vram - factorized_param_vram
static_pct_saved = (static_vram_saved / unprocessed_param_vram) * 100

print(f"\n{'='*95}")
print(f"STRUCTURAL PHYSICAL REPLACEMENT COMPLETE ACROSS ALL 78 PROJECTIONS")
print(f"{'='*95}")
print(f"Pristine Static Parameter VRAM:     {unprocessed_param_vram:.2f} MB")
print(f"Factorized Static Parameter VRAM:   {factorized_param_vram:.2f} MB")
print(f"Net Physical GPU Memory Saved:      {static_vram_saved:.2f} MB ({static_pct_saved:.2f}% of total model weights!)")
print(f"Total Parameters Eliminated:        349,832,340")
print(f"{'='*95}")


In [ ]:
# =====================================================================
# STEP 8: Factorized Model Inference (1,000 Samples) & Peak VRAM Profiling
# =====================================================================
clear_cuda_cache()

factorized_preds = []
model.eval()
print(f"Running Factorized Model GLUE MNLI evaluation across {NUM_EVAL_SAMPLES} samples...")

with torch.no_grad():
    for sample in tqdm(eval_data, desc="Factorized MNLI Inference (1000 samples)"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        factorized_preds.append(pred_label)

factorized_snapshot = get_cuda_memory_snapshot()
factorized_accuracy = accuracy_score(ground_truths, factorized_preds)
factorized_peak_vram = factorized_snapshot["max_allocated_mb"]
accuracy_delta = factorized_accuracy - pristine_accuracy
peak_vram_saved = pristine_peak_vram - factorized_peak_vram

print(f"\n{'='*95}")
print(f"FACTORIZED MODEL 1,000-SAMPLE EVALUATION RESULTS (Notebook 02 Method):")
print(f"{'='*95}")
print(f"  Downstream Accuracy:         {factorized_accuracy * 100:.2f}% (Δ vs Pristine: {accuracy_delta * 100:+.2f}%)")
print(f"  Static Parameter VRAM:       {factorized_param_vram:.2f} MB (Saved: {static_vram_saved:.2f} MB)")
print(f"  Peak Runtime Inference VRAM: {factorized_peak_vram:.2f} MB (Saved: {peak_vram_saved:.2f} MB)")
print(f"{'='*95}")


In [ ]:
# =====================================================================
# STEP 9: Qualitative Generation: Factorized Model Chocolate Cake Recipe
# =====================================================================
print(f"\n{'='*95}")
print("Generating Recipe with Factorized Inactive [Notebook 02 Method] Model:")
print(f"{'='*95}")

cake_inputs = tokenizer(cake_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    factorized_cake_tokens = model.generate(
        **cake_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
factorized_cake_recipe = tokenizer.decode(
    factorized_cake_tokens[0][cake_inputs.input_ids.shape[1]:], skip_special_tokens=True
)
print(factorized_cake_recipe)
print(f"{'='*95}")


In [ ]:
# =====================================================================
# STEP 10: Comprehensive VRAM & Generation Summary Table & Artifact Export
# =====================================================================
print(f"\n{'='*115}")
print(f"{'Metric / Dimension':<45} | {'Pristine Baseline':<20} | {'Factorized [02 Method]':<25} | {'Net Savings / Delta':<18}")
print(f"{'='*115}")
print(f"{'Static Model Parameter VRAM (MB)':<45} | {unprocessed_param_vram:>17.2f} MB | {factorized_param_vram:>22.2f} MB | {static_vram_saved:>14.2f} MB ({static_pct_saved:.1f}%)")
print(f"{'Peak Runtime Inference VRAM (MB)':<45} | {pristine_peak_vram:>17.2f} MB | {factorized_peak_vram:>22.2f} MB | {peak_vram_saved:>14.2f} MB")
print(f"{'Parameters Cut':<45} | {'0':>20} | {'349,832,340':>25} | {'-349,832,340':>18}")
print(f"{'GLUE MNLI Accuracy (1,000 samples)':<45} | {pristine_accuracy*100:>19.2f}% | {factorized_accuracy*100:>24.2f}% | {accuracy_delta*100:>+17.2f}%")
print(f"{'='*115}")

print(f"\n{'='*115}")
print("QUALITATIVE RECIPE COMPARISON: 'What is the best recipe to make a chocolate cake?'")
print(f"{'='*115}")
print(f"--- [Pristine Model Output (First 350 chars)] ---")
print(pristine_cake_recipe[:350] + "...")
print(f"\n--- [Factorized Inactive [Notebook 02 Method] Model Output (First 350 chars)] ---")
print(factorized_cake_recipe[:350] + "...")
print(f"{'='*115}")

artifacts_dir = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("./artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "03_vram_inactive_tucker_1000_samples_results.json"

payload = {
    "experiment": "03_vram_profiling_inactive_tucker_1000_samples_02_method",
    "target_model": MODEL_ID,
    "ranks": RANKS_02,
    "num_layers": NUM_LAYERS,
    "num_projections": 78,
    "num_eval_samples": NUM_EVAL_SAMPLES,
    "unprocessed_param_vram_mb": unprocessed_param_vram,
    "factorized_param_vram_mb": factorized_param_vram,
    "static_vram_saved_mb": static_vram_saved,
    "static_vram_reduction_pct": round(static_pct_saved, 2),
    "baseline_peak_vram_mb": pristine_peak_vram,
    "factorized_peak_vram_mb": factorized_peak_vram,
    "peak_vram_saved_mb": peak_vram_saved,
    "params_eliminated": 349832340,
    "pristine_accuracy_1000": round(pristine_accuracy * 100, 2),
    "factorized_accuracy_1000": round(factorized_accuracy * 100, 2),
    "accuracy_delta_1000": round(accuracy_delta * 100, 2),
    "recipe_prompt": cake_prompt,
    "pristine_cake_recipe": pristine_cake_recipe,
    "factorized_cake_recipe": factorized_cake_recipe,
}

with open(results_file, "w") as f:
    json.dump(payload, f, indent=2)

print(f"\nSaved Experiment 03 results to {results_file}")
